# Upload Files — Interactive Notebook

Companion notebook for [`upload_files.robot`](./upload_files.robot).
Each section mirrors **one row** of the Robot test's `[Template]` block, but uses
raw `requests` calls against the SnapLogic SLFS API so you can see exactly what the
Robot keyword `Upload Files To SnapLogic From Template` does behind the scenes.

🔍 **Robot vs Notebook** — The Robot keyword wraps wildcard expansion, multipart
upload, and result aggregation. This notebook is a **simplified educational
version** showing the core flow. For production tests, use the Robot keyword.

## What this notebook covers

1. **Setup** — load env vars, build a SnapLogic API session
2. **Helpers** — pattern matching, upload, list, delete
3. **Row 1** — Single file upload (`test.expr`)
4. **Row 2** — Wildcard upload (`*.expr`)
5. **Bonus** — Single-character wildcard (`?` pattern)
6. **Verify** — list files in the destination path
7. **Cleanup** — delete the uploaded files (optional)

## Prerequisites

- Start Jupyter via `make jupyter-start` (or `make start-services`).
- Required env vars (auto-loaded by the Setup cell from `/app/.env`):
  - `URL`, `ORG_ADMIN_USER`, `ORG_ADMIN_PASSWORD`, `ORG_NAME`
  - `ACCOUNT_LOCATION_PATH` (destination path, e.g. `test10_project_space/shared`)
- Source files exist under `/app/test/suite/test_data/actual_expected_data/expression_libraries/`.

## SnapLogic SLFS API used here

| Operation | Method | Endpoint |
|---|---|---|
| Upload  | `POST`   | `/api/1/rest/slfs/{ORG}/{dest_path}/{file_name}` (multipart `file=<handle>`) |
| List    | `GET`    | `/api/1/rest/asset/list/{ORG}/{dest_path}` (filter `asset_type="File"`) |
| Delete  | `DELETE` | `/api/1/rest/slfs/{ORG}/{dest_path}/{file_name}?soft_delete=true` |

## Section 1 — Setup

Load credentials, validate they're present, and create a `requests.Session` with Basic Auth.

In [ ]:
import os
import fnmatch
from pathlib import Path
import requests
from dotenv import load_dotenv

# --- Load env vars from root .env --------------------------------------------
# Root .env is already injected by Docker Compose (env_file:) but reloading is
# harmless. The upload tests don't need any database-specific env files.
load_dotenv("/app/.env", override=False)

# --- SnapLogic API credentials -----------------------------------------------
SL_URL  = os.environ.get("URL", "https://elastic.snaplogic.com")
SL_USER = os.environ.get("ORG_ADMIN_USER")
SL_PASS = os.environ.get("ORG_ADMIN_PASSWORD")
ORG     = os.environ.get("ORG_NAME")
DEST    = os.environ.get("ACCOUNT_LOCATION_PATH")

missing = [k for k, v in {
    "ORG_ADMIN_USER": SL_USER, "ORG_ADMIN_PASSWORD": SL_PASS,
    "ORG_NAME": ORG, "ACCOUNT_LOCATION_PATH": DEST,
}.items() if not v]
if missing:
    raise RuntimeError(f"Missing required env vars: {missing}. Source your .env files first.")

# --- Reusable session with Basic auth ----------------------------------------
# Note: multipart uploads must NOT set Content-Type — requests builds it.
session = requests.Session()
session.auth = (SL_USER, SL_PASS)

print(f"✅ SnapLogic session ready")
print(f"   URL:         {SL_URL}")
print(f"   Org:         {ORG}")
print(f"   User:        {SL_USER}")
print(f"   Destination: {ORG}/{DEST}")

## Section 2 — Helper functions

These mirror what `Upload Files To SnapLogic From Template` does internally:

1. **`find_files`** — local pattern matching (mirrors Robot's `Find Files With Pattern`, supports `*` and `?`)
2. **`upload_one_file`** — POST a single file via multipart to `/api/1/rest/slfs/...`
3. **`upload_files`** — batch wrapper that uploads a list and returns per-file results (mirrors Robot's `Upload Files`)
4. **`upload_from_template`** — top-level orchestrator (mirrors Robot's `Upload Files To SnapLogic From Template`)
5. **`list_files`** — list `File` assets at a destination path
6. **`file_exists`** — check if a named file is already at a destination
7. **`delete_file`** — DELETE a file at `<dest>/<name>` (soft-delete by default)

In [ ]:
# Source dir used by upload_files.robot (resolved from the Robot's ${CURDIR}/../../../../test_data/...)
SOURCE_DIR = Path("/app/test/suite/test_data/actual_expected_data/expression_libraries")


def find_files(source_dir: Path, pattern: str) -> list[Path]:
    """Find files in source_dir matching pattern (supports * and ?).

    Mirrors Robot's `Find Files With Pattern`. Non-recursive; only files
    directly in source_dir are considered.
    """
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Source directory not found: {source_dir}")
    matches = sorted(
        p for p in source_dir.iterdir()
        if p.is_file() and fnmatch.fnmatch(p.name, pattern)
    )
    return matches


def upload_one_file(dest_path: str, file_path: Path, upload_name: str | None = None) -> dict:
    """POST a single file to SLFS via multipart upload.

    Returns a result dict shaped like Robot's `Upload Files` per-file result:
    {file, uploaded_as, destination, success, message}.
    Never raises — failures are captured in `success=False` / `message`.
    """
    upload_name = upload_name or file_path.name
    result = {
        "file": str(file_path), "uploaded_as": upload_name,
        "destination": f"{ORG}/{dest_path}", "success": False, "message": "",
    }
    if not file_path.is_file():
        result["message"] = f"File not found: {file_path}"
        return result
    url = f"{SL_URL}/api/1/rest/slfs/{ORG}/{dest_path}/{upload_name}"
    try:
        with open(file_path, "rb") as fh:
            resp = session.post(url, files={"file": (upload_name, fh)})
        if resp.ok:
            result["success"] = True
            result["message"] = f"Upload successful: {upload_name}"
        else:
            result["message"] = f"Upload failed [{resp.status_code}]: {resp.text[:300]}"
    except Exception as exc:
        result["message"] = f"Upload exception: {exc}"
    return result


def upload_files(file_paths: list[Path], dest_path: str) -> list[dict]:
    """Sequentially upload a list of files. Returns list of per-file result dicts.

    Mirrors Robot's `Upload Files` keyword (without rename_to support).
    """
    return [upload_one_file(dest_path, fp) for fp in file_paths]


def upload_from_template(source_dir: Path, file_name: str, dest_path: str) -> list[dict]:
    """Top-level orchestrator — mirrors `Upload Files To SnapLogic From Template`.

    Expands wildcards, uploads each match, prints a summary, raises if any failed.
    """
    if "*" in file_name or "?" in file_name:
        files = find_files(source_dir, file_name)
    else:
        files = [source_dir / file_name]
    print(f"📤 Uploading {len(files)} file(s) matching '{file_name}' from {source_dir}")
    print(f"   → destination: {ORG}/{dest_path}")
    results = upload_files(files, dest_path)
    ok    = [r for r in results if r["success"]]
    fails = [r for r in results if not r["success"]]
    for r in results:
        prefix = "   ✓" if r["success"] else "   ✗"
        print(f"{prefix} {r['uploaded_as']:35s}  {r['message']}")
    if fails:
        raise AssertionError(f"{len(fails)} of {len(results)} upload(s) failed")
    print(f"✅ All {len(ok)} file(s) uploaded successfully")
    return results


def list_files(dest_path: str) -> list[dict]:
    """List File assets at the destination path."""
    url = f"{SL_URL}/api/1/rest/asset/list/{ORG}/{dest_path}"
    r = session.get(url)
    r.raise_for_status()
    entries = r.json().get("response_map", {}).get("entries", [])
    return [e for e in entries if e.get("asset_type") == "File"]


def file_exists(dest_path: str, file_name: str) -> dict | None:
    """Return the matching file entry if it exists, else None."""
    for entry in list_files(dest_path):
        if entry["name"] == file_name:
            return entry
    return None


def delete_file(dest_path: str, file_name: str, soft_delete: bool = True) -> None:
    """DELETE a file at <dest>/<name>.

    soft_delete=True (default) marks as deleted but keeps for recovery;
    soft_delete=False removes permanently.
    """
    url = f"{SL_URL}/api/1/rest/slfs/{ORG}/{dest_path}/{file_name}"
    r = session.delete(url, params={"soft_delete": str(soft_delete).lower()})
    if not r.ok:
        print(f"❌ {r.status_code} response body: {r.text[:500]}")
    r.raise_for_status()


print("✅ Helper functions defined")

## Section 3 — Row 1: Single file upload

Mirrors this Robot test row:

```robot
${CURDIR}/../../../../test_data/actual_expected_data/expression_libraries    test.expr    ${ACCOUNT_LOCATION_PATH}
```

First call → file is uploaded. Re-running this cell overwrites the file in SLFS
(SLFS upload is idempotent — same filename gets replaced).

In [ ]:
results_row1 = upload_from_template(
    source_dir=SOURCE_DIR,
    file_name="test.expr",
    dest_path=DEST,
)

## Section 4 — Row 2: Wildcard upload (`*.expr`)

Mirrors this Robot test row:

```robot
${upload_source_file_path}    *.expr    ${ACCOUNT_LOCATION_PATH}
```

Expands to every `.expr` file directly in the source directory.

In [ ]:
results_row2 = upload_from_template(
    source_dir=SOURCE_DIR,
    file_name="*.expr",
    dest_path=DEST,
)

## Section 5 — Bonus: Single-character wildcard (`?` pattern)

Mirrors the commented-out example in the Robot test:

```robot
# ${upload_source_file_path}    employees.?pr    ${ACCOUNT_LOCATION_PATH}
```

Demonstrates `?` matching a single character. For our source dir, `test.?xpr`
matches `test.expr` (just one file).

In [ ]:
results_row3 = upload_from_template(
    source_dir=SOURCE_DIR,
    file_name="test.?xpr",
    dest_path=DEST,
)

## Section 6 — Verify (list files at destination)

Confirms the previous cells uploaded what they claim. Uses the same
`/asset/list/...` endpoint as the create-account notebook, filtered to
`asset_type == "File"`.

In [ ]:
uploaded = list_files(DEST)
print(f"📋 Files in {ORG}/{DEST}: ({len(uploaded)} total)")
expected_names = {p.name for p in find_files(SOURCE_DIR, "*.expr")}
for entry in uploaded:
    asset_id = (entry.get("asset_id") or "")[:16]
    marker = " ⭐ uploaded above" if entry["name"] in expected_names else ""
    print(f"   - {entry['name']:35s}  ({asset_id}...){marker}")

# Sanity check — every file from Row 2's wildcard should be visible now
found_names = {e["name"] for e in uploaded}
missing = expected_names - found_names
if missing:
    print(f"\n⚠️  Missing from SLFS: {missing}")
else:
    print(f"\n✅ All {len(expected_names)} expected .expr files present in SLFS")

## Section 7 — Cleanup (optional)

Delete the uploaded test files so the destination is left clean. Skip this
if you want to inspect the files in the SnapLogic Manager UI.

`soft_delete=True` (default) sends them to a recoverable trash;
`soft_delete=False` removes them permanently.

In [ ]:
for entry in list_files(DEST):
    if entry["name"] in expected_names:
        delete_file(DEST, entry["name"], soft_delete=True)
        print(f"🗑️  deleted {DEST}/{entry['name']}")

remaining = {e["name"] for e in list_files(DEST)} & expected_names
if remaining:
    print(f"\n⚠️  Still present after delete: {remaining}")
else:
    print(f"\n✅ All test files removed from {DEST}")

## What's next

- **Tweak any cell** — change patterns (`*.json`, `app_2025_??.log`), point at a different `source_dir`, or try `soft_delete=False`.
- **Inspect in SnapLogic Manager** — log into the SnapLogic Manager UI to see the uploaded files under `<ORG> → <project_space> → shared`.
- **Run the official Robot test instead** for production validation:
  ```bash
  make robot-run-tests-no-gp TAGS="upload_file_sample"
  ```

## Caveats vs the Robot keyword

This notebook is a simplified educational version. The Robot keyword `Upload Files To SnapLogic From Template` additionally:

- Supports the `file:///...` URL form for `source_dir` (e.g. JAR uploads)
- Provides Robot-style logging, tags, and detailed `Log` console output
- Integrates with `Find Files With Pattern` which also handles edge cases like Windows path separators
- Hooks into the broader suite's `${ORG_ADMIN_SESSION}` for auth-mode handling (Basic vs OAuth)

For production tests, **use the Robot keyword**. This notebook is for learning what's underneath.

## Related

- [`upload_files.robot`](./upload_files.robot) — runnable Robot test (2 rows + commented examples)
- [`../01.accounts/create_account.ipynb`](../01.accounts/create_account.ipynb) — sibling notebook for account creation